# NN_Ibd — SiC MOSFET Body-Diode Hybrid Model

This notebook implements the paper equations shown in the supplied figure:

\[
I(t)=\frac{q_E-q_M}{T_M} \tag{26}
\]

\[
0=\frac{dq_M}{dt}+\frac{q_M}{\tau}-\frac{q_E-q_M}{T_M} \tag{27}
\]

\[
q_E=\tau I_{bd}(V_r) \tag{28}
\]

and replaces the static current expression by

\[
I_{bd}=f_{NN}(V_{gs},V_{ds}) \tag{29}
\]

The ANN structure is exactly the one stated in the excerpt:

**2 inputs → 6 neurons → 6 neurons → 1 output**

The excerpt does not state the activation function. `Tanh` is used here to keep the fitted compact-model curve smooth.

## Cell 1 — Imports and device

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from body_diode_hybrid import (
    BodyDiodeNN,
    LumpedChargeParameters,
    load_ibd_csv,
    train_ibd_network,
    predict_ibd,
    save_ibd_model,
    simulate_hybrid_body_diode,
)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device =", device)

## Cell 2 — Configuration
Change only the file path / column names if your measured CSV uses different names.

In [ ]:
DATA_FILE = Path("Ibd_data.csv")

VGS_COL = "Vgs"
VDS_COL = "Vds"
IBD_COL = "Ibd"

EPOCHS = 5000
LEARNING_RATE = 1e-3
BATCH_SIZE = 128
VAL_FRACTION = 0.20
SEED = 42

MODEL_FILE = Path("NN_Ibd.pth")

print("DATA_FILE =", DATA_FILE.resolve())

## Cell 3 — Load measured static body-diode data
Required CSV columns: `Vgs, Vds, Ibd`.

In [ ]:
df, Vgs, Vds, Ibd = load_ibd_csv(
    DATA_FILE,
    vgs_col=VGS_COL,
    vds_col=VDS_COL,
    ibd_col=IBD_COL,
)

print(df.head())
print()
print("number of samples =", len(df))
print()
print(df.describe())

## Cell 4 — Inspect the measured data range

In [ ]:
print(
    f"Vgs range: {df[VGS_COL].min():.6g} V "
    f"to {df[VGS_COL].max():.6g} V"
)

print(
    f"Vds range: {df[VDS_COL].min():.6g} V "
    f"to {df[VDS_COL].max():.6g} V"
)

print(
    f"Ibd range: {df[IBD_COL].min():.6g} A "
    f"to {df[IBD_COL].max():.6g} A"
)

## Cell 5 — Create the paper-required 2–6–6–1 network

In [ ]:
model_preview = BodyDiodeNN()

print(model_preview)

## Cell 6 — Train Eq. (29): Ibd = fNN(Vgs, Vds)

In [ ]:
model, norm, history = train_ibd_network(
    Vgs,
    Vds,
    Ibd,
    device=device,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    val_fraction=VAL_FRACTION,
    seed=SEED,
    print_every=250,
)

print()
print("Best validation loss =", history.best_val_loss)

## Cell 7 — Plot training and validation loss

In [ ]:
plt.figure(figsize=(7, 5))

plt.semilogy(
    history.train_loss,
    label="train"
)

plt.semilogy(
    history.val_loss,
    label="validation"
)

plt.xlabel("Epoch")
plt.ylabel("Normalized MSE")
plt.title("NN_Ibd Training Loss")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Cell 8 — Predict all measured points

In [ ]:
Ibd_pred = predict_ibd(
    model,
    norm,
    Vgs.reshape(-1),
    Vds.reshape(-1),
    device=device,
).cpu().numpy().reshape(-1)

Ibd_true = Ibd.numpy().reshape(-1)

print("prediction points =", len(Ibd_pred))

## Cell 9 — Static-model error metrics

In [ ]:
error = Ibd_pred - Ibd_true

rmse = np.sqrt(
    np.mean(error ** 2)
)

mae = np.mean(
    np.abs(error)
)

ss_res = np.sum(
    (Ibd_true - Ibd_pred) ** 2
)

ss_tot = np.sum(
    (Ibd_true - np.mean(Ibd_true)) ** 2
)

r2 = (
    1.0 - ss_res / ss_tot
    if ss_tot > 0
    else np.nan
)

print(f"RMSE = {rmse:.8g} A")
print(f"MAE  = {mae:.8g} A")
print(f"R^2  = {r2:.8f}")

## Cell 10 — Measured Ibd vs ANN-predicted Ibd

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(
    Ibd_true,
    Ibd_pred,
    s=20
)

lo = min(
    Ibd_true.min(),
    Ibd_pred.min()
)

hi = max(
    Ibd_true.max(),
    Ibd_pred.max()
)

plt.plot(
    [lo, hi],
    [lo, hi],
    "--"
)

plt.xlabel("Measured Ibd [A]")
plt.ylabel("Predicted Ibd [A]")
plt.title("Static Body-Diode ANN")
plt.grid(True)
plt.axis("equal")
plt.tight_layout()
plt.show()

## Cell 11 — Third-quadrant static curves for each measured Vgs

In [ ]:
vgs_values = np.sort(
    df[VGS_COL].unique()
)

plt.figure(figsize=(8, 6))

for vgs_value in vgs_values:

    sub = df[
        np.isclose(
            df[VGS_COL].to_numpy(),
            vgs_value
        )
    ].sort_values(VDS_COL)

    vds_curve = sub[
        VDS_COL
    ].to_numpy(dtype=float)

    ibd_measured_curve = sub[
        IBD_COL
    ].to_numpy(dtype=float)

    vgs_curve = np.full(
        len(vds_curve),
        vgs_value,
        dtype=float
    )

    ibd_pred_curve = predict_ibd(
        model,
        norm,
        vgs_curve,
        vds_curve,
        device=device,
    ).cpu().numpy().reshape(-1)

    plt.scatter(
        vds_curve,
        ibd_measured_curve,
        s=18
    )

    plt.plot(
        vds_curve,
        ibd_pred_curve,
        label=f"Vgs={vgs_value:g} V"
    )

plt.xlabel("Vds [V]")
plt.ylabel("Ibd [A]")
plt.title("Vgs-Dependent Body-Diode Static Characteristic")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Cell 12 — Save the trained NN_Ibd model

In [ ]:
save_ibd_model(
    MODEL_FILE,
    model,
    norm,
)

print("Saved model:", MODEL_FILE.resolve())

## Cell 13 — Lumped-charge parameters

The paper defines:

- \(T_M\): transit time
- \(\tau\): carrier lifetime

These are **device parameters**, not universal constants. Replace the example initial values below with identified/fitted values for your device.

In [ ]:
tau = 100e-9   # [s] example placeholder
TM  = 20e-9    # [s] example placeholder

params = LumpedChargeParameters(
    tau=tau,
    TM=TM
)

params.validate()

print("tau =", params.tau, "s")
print("TM  =", params.TM, "s")

## Cell 14 — Define a transient input waveform

For a real reverse-recovery simulation, replace these arrays with measured/simulated `Vgs(t)` and `Vds(t)` from your switching test.

Do not trust ANN extrapolation far outside the Vgs/Vds range printed in Cell 4.

In [ ]:
dt = 0.2e-9
t_end = 400e-9

time_s = np.arange(
    0.0,
    t_end + dt,
    dt
)

# Demonstration only.
# Keep these values inside your trained static-data voltage domain
# when you use the model quantitatively.
Vgs_TRANSIENT = float(
    df[VGS_COL].iloc[0]
)

Vds_forward = float(
    df[VDS_COL].iloc[
        np.argmax(
            np.abs(
                df[IBD_COL].to_numpy()
            )
        )
    ]
)

Vds_after = float(
    df[VDS_COL].iloc[
        np.argmin(
            np.abs(
                df[IBD_COL].to_numpy()
            )
        )
    ]
)

switch_time = 150e-9

Vgs_waveform = np.full(
    len(time_s),
    Vgs_TRANSIENT
)

Vds_waveform = np.where(
    time_s < switch_time,
    Vds_forward,
    Vds_after
)

print("Transient Vgs =", Vgs_TRANSIENT, "V")
print("Vds before switch =", Vds_forward, "V")
print("Vds after switch  =", Vds_after, "V")

## Cell 15 — Run the complete hybrid model: Eqs. (26)–(29)

In [ ]:
result = simulate_hybrid_body_diode(
    model,
    norm,
    time_s,
    Vgs_waveform,
    Vds_waveform,
    params,
    device=device,
    initial_qM=None,
    integration="rk4",
)

print(result.keys())

## Cell 16 — Plot qE and qM

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    result["time_s"] * 1e9,
    result["qE_C"],
    label="qE"
)

plt.plot(
    result["time_s"] * 1e9,
    result["qM_C"],
    label="qM"
)

plt.xlabel("Time [ns]")
plt.ylabel("Charge [C]")
plt.title("Lumped-Charge Model")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Cell 17 — Plot static Ibd and transient I(t)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    result["time_s"] * 1e9,
    result["Ibd_static_A"],
    "--",
    label="Ibd = fNN(Vgs,Vds)"
)

plt.plot(
    result["time_s"] * 1e9,
    result["I_transient_A"],
    label="I(t) from Eq. (26)"
)

plt.xlabel("Time [ns]")
plt.ylabel("Current [A]")
plt.title("Hybrid Body-Diode Model")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Cell 18 — Model equations implemented

The notebook now implements exactly this chain:

\[
(V_{gs},V_{ds})
\overset{f_{NN}}{\longrightarrow}
I_{bd}
\]

\[
q_E=\tau I_{bd}
\]

\[
\frac{dq_M}{dt}
=
\frac{q_E-q_M}{T_M}
-\frac{q_M}{\tau}
\]

\[
I(t)=\frac{q_E-q_M}{T_M}
\]

The ANN structure follows the paper excerpt exactly: **two hidden layers, six neurons per hidden layer**.